In [0]:
dbutils.widgets.text("sas_token", "", "Azure Storage SAS Token")

In [0]:
import requests
import json
import xml.etree.ElementTree as ET
from pyspark.sql.functions import col, current_timestamp, expr
import os

def get_sas_token() -> str:
    token = os.getenv("AZURE_SAS_TOKEN", "").strip()
    
    if not token:
        token = dbutils.widgets.get("sas_token").strip()
        
    if not token:
        raise ValueError(
            "SAS Token não informado. Preencha o widget 'Azure Storage SAS Token' "
            "no topo do notebook ou defina a variável de ambiente AZURE_SAS_TOKEN."
        )
        
    return token.lstrip("?")

storage_account = "stdataengprojectrafael"
container = "bronze"
prefix = "world-bank/population/year=2026/month=11/day=09/"
sas_token = get_sas_token()

list_url = f"https://{storage_account}.blob.core.windows.net/{container}?restype=container&comp=list&prefix={prefix}&{sas_token}"
resp_list = requests.get(list_url)
resp_list.raise_for_status()

root = ET.fromstring(resp_list.content)
blob_names = sorted([
    blob.find("Name").text
    for blob in root.findall(".//Blob")
    if blob.find("Name") is not None and blob.find("Name").text.endswith(".json")
])
print(f"Arquivos encontrados ({len(blob_names)}):", blob_names)

decoder = json.JSONDecoder()
all_records = []

for blob_name in blob_names:
    blob_url = f"https://{storage_account}.blob.core.windows.net/{container}/{blob_name}?{sas_token}"
    resp_data = requests.get(blob_url)
    resp_data.raise_for_status()

    text = resp_data.text.strip()
    idx = 0
    n = len(text)
    while idx < n:
        while idx < n and text[idx].isspace():
            idx += 1
        if idx >= n:
            break
        obj, idx = decoder.raw_decode(text, idx)

        if isinstance(obj, list):
            if len(obj) >= 2 and isinstance(obj[1], list):
                all_records.extend(obj[1])
            else:
                all_records.extend([r for r in obj if isinstance(r, dict) and "country" in r])
        elif isinstance(obj, dict) and "country" in obj:
            all_records.append(obj)

print(f"Total de registros brutos empilhados: {len(all_records)}")

df_raw = spark.createDataFrame(all_records)

df_silver = (
    df_raw
    .select(
        col("country.id").alias("country_code"),
        col("country.value").alias("country_name"),
        col("countryiso3code").alias("country_iso3"),
        col("indicator.id").alias("indicator_id"),
        col("indicator.value").alias("indicator_name"),
        expr("try_cast(date as int)").alias("year"),
        expr("try_cast(value as long)").alias("population"),
        current_timestamp().alias("silver_processed_at")
    )
    .filter(col("population").isNotNull())
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.worldbank_population")
)

display(spark.sql("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(DISTINCT country_code) AS total_codigos_iso2,
        COUNT(DISTINCT country_iso3) AS total_codigos_iso3,
        MIN(year) AS ano_inicial,
        MAX(year) AS ano_final
    FROM silver.worldbank_population
"""))